# 第11章 指示チューニング

## 11.3 指示チューニングしたモデルの評価

In [ ]:
# Apple Silicon (Mac) ではbitsandbytesが使えないため、量子化は使わずfp16で動かす
!pip install flexeval

### 11.3.1 モデルの動作確認

In [ ]:
from flexeval import HuggingFaceLM

model_name = "llm-book/Swallow-7b-hf-oasst1-21k-ja"
# Apple SiliconのGPU(MPS)とfp16を明示的に指定してロードする
# （指定しないとCPUにロードされ、7Bモデルでは極端に遅くなる）
# なお、HuggingFaceLMは遅延ロードのため、実際の読み込みは次セルの初回生成時に行われる
llm = HuggingFaceLM(
    model=model_name,
    model_kwargs={"device_map": "mps", "dtype": "float16"},
)

In [ ]:
input_messages = [{"role": "user", "content": "1+1はなんでしょうか？"}]
print(llm.generate_chat_response(input_messages))
# 初回生成でモデルがロードされるので、ここでデバイスを確認できる（mpsならOK）
print("model device:", next(llm.model.parameters()).device)

### 11.3.2 指示追従性能の評価

In [ ]:
import gc
import torch

# MPSに載せたモデルをCPUに移し、GPUメモリを解放する
# HuggingFaceLMは遅延ロードのため、動作確認をしていない場合はllm.modelがNoneになる
if llm.model is not None:
    llm.model.cpu()
del llm
gc.collect()
torch.mps.empty_cache()

In [ ]:
from pathlib import Path

# 評価結果のローカル保存先を作成（Colabのドライブマウントは不要）
save_dir = "./outputs/IT_eval/vicuna-ja"
Path(save_dir).mkdir(parents=True, exist_ok=True)

In [ ]:
# Apple Silicon (MPS) での評価コマンド
# bitsandbytesはMPS非対応のため量子化は使わず、fp16でロードする
# PYTORCH_ENABLE_MPS_FALLBACK=1 でMPS未実装オペレータをCPUにフォールバックさせる
!PYTORCH_ENABLE_MPS_FALLBACK=1 flexeval_lm \
  --language_model HuggingFaceLM \
  --language_model.model "llm-book/Swallow-7b-hf-oasst1-21k-ja" \
  --language_model.model_kwargs.device_map "mps" \
  --language_model.model_kwargs.dtype "float16" \
  --eval_setup "vicuna-ja" \
  --eval_setup.gen_kwargs '{do_sample: True, temperature: 0.7, top_p: 0.9, max_new_tokens: 1024}' \
  --eval_setup.batch_size 1 \
  --save_dir "./outputs/IT_eval/vicuna-ja" \
  --force true

In [ ]:
import json
from pathlib import Path

save_dir = "./outputs/IT_eval/vicuna-ja"

with open(Path(save_dir) / "outputs.jsonl") as f:
    for line in f:
        item = json.loads(line)
        print("===== 入力 ====")
        print(item["task_inputs"]["messages"][0]["content"])
        print("===== モデル出力 ====")
        print(item["lm_output"])
        break   # 全件確認する場合は消してください

In [ ]:
!flexeval_presets assistant_eval_ja_single_turn

In [ ]:
import nest_asyncio
from flexeval import instantiate_from_config

# assistant_eval_ja_single_turnの設定ファイルからMetricをインスタンス化
